In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import logging

logging.basicConfig(level=logging.ERROR)

In [12]:
from datasets import load_dataset, DownloadConfig

download_config = DownloadConfig(
    local_files_only=True,
    cache_dir=".cache",   # optional
)

LANGS = ['af_za', 'am_et', 'ar_eg', 'ast_es', 'az_az', 'be_by', 'bg_bg', 'bn_in', 'ca_es', 'ceb_ph', 'ckb_iq', 'cmn_hans_cn', 'cs_cz', 'cy_gb', 'da_dk', 'de_de', 'el_gr', 'en_us', 'es_419', 'et_ee', 'fa_ir', 'ff_sn', 'fi_fi', 'fr_fr', 'ga_ie', 'gl_es', 'ha_ng', 'he_il', 'hi_in', 'hr_hr', 'hu_hu', 'hy_am', 'id_id', 'it_it', 'ja_jp', 'jv_id', 'ka_ge', 'kk_kz', 'km_kh', 'kn_in', 'ko_kr', 'ky_kg', 'lg_ug', 'lo_la', 'lt_lt', 'lv_lv', 'mi_nz', 'mk_mk', 'ml_in', 'mn_mn', 'mr_in', 'ms_my', 'mt_mt', 'my_mm', 'nb_no', 'ne_np', 'nl_nl', 'ny_mw', 'om_et', 'or_in', 'pa_in', 'pl_pl', 'ps_af', 'pt_br', 'ro_ro', 'ru_ru', 'sl_si', 'sn_zw', 'so_so', 'sv_se', 'sw_ke', 'ta_in', 'te_in', 'tg_tj', 'th_th', 'tr_tr', 'uk_ua', 'ur_pk', 'uz_uz', 'vi_vn', 'wo_sn', 'xh_za', 'yo_ng', 'yue_hant_hk', 'zu_za']

In [13]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC

model_path = "models/mms-300m-ipa/"
processor = Wav2Vec2Processor.from_pretrained(model_path)
model = Wav2Vec2ForCTC.from_pretrained(model_path)
tokenizer = processor.tokenizer

In [21]:
from typing import List, Optional
import ctc_segmentation
import numpy as np
from transformers import Wav2Vec2Tokenizer
from lingpy.sequence.sound_classes import ipa2tokens

    
class CTCSegmentation:

    def __init__(self, tokenizer: Wav2Vec2Tokenizer, sampling_rate: int= 16000):
        self.tokenizer = tokenizer
        char_list = [tokenizer.convert_ids_to_tokens(i) for i in range(tokenizer.vocab_size)]
        self.config = ctc_segmentation.CtcSegmentationParameters(char_list=char_list)
        self.sampling_rate = sampling_rate
        

    def get_word_and_timestamps_batch(
        self,
        probs_batch: List[np.ndarray],
        audio_lens: List[int],
        frame_lens: List[int],
        ipa_transcripts: Optional[List[str]],
        transcripts: Optional[List[str]],
        sample_rate: int = 16000,
    ) -> List[dict]:

        alignments = []
        
        
        # --- Transcript handling ---
        if not ipa_transcripts:
            pred_ids = probs_batch.argmax(axis=-1)
            ipa_transcripts = self.tokenizer.batch_decode(pred_ids)

        if not transcripts:
            transcripts = ipa_transcripts
            
        for probs, audio_len, frame_len, ipa_transcript, transcript in zip(probs_batch, audio_lens, frame_lens, ipa_transcripts, transcripts):
            
            probs = probs[:frame_len]
            self.config.index_duration = ( audio_len / self.sampling_rate ) / frame_len
            # =====================
            # WORD ALIGNMENT
            # =====================
            words_ipa = ipa_transcript.split()
            words = transcript.split()

            assert len(words) == len(words_ipa)
            
            gt_mat, utt_idx = ctc_segmentation.prepare_text(self.config, words_ipa)

            try:
                timings, char_probs, _ = ctc_segmentation.ctc_segmentation(
                    self.config, probs, gt_mat
                )
    
                word_segments = ctc_segmentation.determine_utterance_segments(
                    self.config, utt_idx, char_probs, timings, words_ipa
                )
            except:
                unit = frame_len*self.config.index_duration/len(words)
                word_segments = [(unit*i, unit*(i+1) ,0.0) for i in range(len(words))]

            word_out = [
                {
                    "text": w,
                    "start": round(float(p[0]),3),
                    "end": round(float(p[1]),3),
                    "conf": round(float(p[2]),3),
                }
                for w, p in zip(words, word_segments)
            ]

            # =====================
            # PHONE ALIGNMENT (hierarchical)
            # =====================
            
            phone_out = []
            
            PAD_FRAMES = 0
            
            for word, word_ipa, seg in zip(words, words_ipa, word_out):
            
                # ---------------------------------
                # convert word timestamps -> frames
                # ---------------------------------
                start_frame = int(
                    seg["start"] / self.config.index_duration
                )
            
                end_frame = int(
                    seg["end"] / self.config.index_duration
                )
            
                # add small context padding
                start_frame = max(0, start_frame - PAD_FRAMES)
                end_frame = min(len(probs), end_frame + PAD_FRAMES)
            
            
                # crop local probabilities
                local_probs = probs[start_frame:end_frame]
            
                # ---------------------------------
                # tokenize IPA word
                # ---------------------------------
                if word_ipa != '<unk>':
                    phones = ipa2tokens(
                        word_ipa,
                        merge_vowels=False
                    )
                else:
                    phones = ['<unk>']
                
            
            
                gt_mat_p, utt_idx_p = ctc_segmentation.prepare_text(
                    self.config,
                    phones
                )
                
                try:
                    timings_p, char_probs_p, _ = (
                        ctc_segmentation.ctc_segmentation(
                            self.config,
                            local_probs,
                            gt_mat_p
                        )
                    )
                
            
                    phone_segments = (
                        ctc_segmentation.determine_utterance_segments(
                            self.config,
                            utt_idx_p,
                            char_probs_p,
                            timings_p,
                            phones
                        )
                    )
                    
                except:
                    unit = max(0.001, (end_frame - start_frame)*self.config.index_duration/len(phones))
                    phone_segments = [(i*unit, (i+1)*unit, 0.0) for i, ph in enumerate(phones)]
            
                # ---------------------------------
                # convert local -> global timestamps
                # ---------------------------------
                word_phone_out = []
            
                for ph, p in zip(phones, phone_segments):
            
                    global_start = (
                        p[0]
                        + start_frame * self.config.index_duration
                    )
            
                    global_end = (
                        p[1]
                        + start_frame * self.config.index_duration
                    )
            
                    global_start = round(float(global_start), 3)
                    global_end = round(float(global_end), 3)
            
                    # monotonic safety
                    if global_end <= global_start:
                        global_end = global_start + 0.001
            
                    word_phone_out.append(
                        {
                            "text": ph,
                            "start": global_start,
                            "end": global_end,
                            "conf": round(float(p[2]), 3),
                        }
                    )
            
                phone_out.extend(word_phone_out)
                

            alignments.append({
                "words": word_out,
                "phones": phone_out,
            })
        return alignments

In [22]:
from praatio import textgrid


def _build_intervals_with_pauses(
    segments,
    xmin,
    xmax,
    min_gap=0.001,
):
    """
    Fill gaps between segments with silence intervals,
    while enforcing:
      - monotonic intervals
      - no overlaps
      - minimum duration/gap

    segments: list of dicts with keys:
        [start, end, text]
    """

    intervals = []

    # sort for safety
    segments = sorted(
        segments,
        key=lambda x: x["start"]
    )

    cur = float(xmin)

    for seg in segments:

        start = round(float(seg["start"]), 3)
        end = round(float(seg["end"]), 3)
        text = seg["text"]

        # ---------------------------------
        # prevent backward movement
        # ---------------------------------
        if start < cur:
            start = cur

        # ---------------------------------
        # enforce minimum duration
        # ---------------------------------
        if end <= start:
            end = start + min_gap

        # ---------------------------------
        # insert silence gap if needed
        # ---------------------------------
        if start - cur >= min_gap:
            intervals.append(
                (
                    round(cur, 3),
                    round(start, 3),
                    ""
                )
            )

        # ---------------------------------
        # add segment
        # ---------------------------------
        intervals.append(
            (
                round(start, 3),
                round(end, 3),
                text
            )
        )

        # next interval must begin AFTER this
        cur = end

    # ---------------------------------
    # tail silence
    # ---------------------------------
    if xmax - cur >= min_gap:

        intervals.append(
            (
                round(cur, 3),
                round(float(xmax), 3),
                ""
            )
        )

    return intervals


def save_textgrids(
    audio_filenames,
    alignments,
    out_dir,
):
    """
    Args:
        audio_paths: List[str]
        alignments: List[dict] with keys "words", "phones"
        out_dir: output directory
    """

    os.makedirs(out_dir, exist_ok=True)

    for filename, align in zip(audio_filenames, alignments):
        words = align["words"]
        phones = align["phones"]

        # determine xmax (end of audio)
        xmax = max(
            max(w["end"] for w in words) if words else 0,
            max(p["end"] for p in phones) if phones else 0,
        )

        xmin = 0.0

        # build tiers (with pauses filled)
        word_intervals = _build_intervals_with_pauses(words, xmin, xmax)
        phone_intervals = _build_intervals_with_pauses(phones, xmin, xmax)

        # create TextGrid
        tg = textgrid.Textgrid()

        word_tier = textgrid.IntervalTier(
            name="words",
            entries=word_intervals,
            minT=xmin,
            maxT=xmax,
        )

        phone_tier = textgrid.IntervalTier(
            name="phones",
            entries=phone_intervals,
            minT=xmin,
            maxT=xmax,
        )

        tg.addTier(word_tier)
        tg.addTier(phone_tier)

        # output filename
        name = os.path.splitext(filename)[0] + ".TextGrid"
        out_path = os.path.join(out_dir, name)

        # save
        tg.save(out_path, format="short_textgrid", includeBlankSpaces=True)

In [23]:
from itertools import islice
from tqdm.auto import tqdm

device = "cuda:0"

model.eval()
model.to(device)


def batch_iterator(iterable, batch_size):
    iterator = iter(iterable)

    while True:
        batch = list(islice(iterator, batch_size))

        if not batch:
            break

        yield batch


output_path = os.path.join("alignments", os.path.basename(model_path) if model_path[-1] != '/' else os.path.basename(model_path[:-1]))
os.makedirs(output_path, exist_ok=True)

In [24]:
import torch

ctc_segmentor = CTCSegmentation(tokenizer=processor.tokenizer)
for lang in tqdm(LANGS):
    dataset = load_dataset("fleurs", lang, streaming=True, download_config=download_config, trust_remote_code=True)
    test_dataset = dataset["test"]   # streaming dataset
    model.load_adapter(lang)
    alignments = []
    audio_filenames = []
    for batch in batch_iterator(test_dataset, batch_size=2):

        # audio arrays
        audios = [x["audio"]["array"][:360000] for x in batch]

        # processor handles padding dynamically
        inputs = processor(
            audios,
            sampling_rate=16000,
            return_tensors="pt",
            padding=True,
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}
        audio_lengths = inputs["attention_mask"].sum(-1)
        logit_lengths = model._get_feat_extract_output_lengths(audio_lengths)

        with torch.no_grad():
            logits = model(**inputs).logits

        probs_batch = torch.softmax(logits, dim=-1)

        alignments.extend(
            ctc_segmentor.get_word_and_timestamps_batch(
                probs_batch=probs_batch.cpu().numpy(),
                audio_lens=audio_lengths.cpu().numpy(),
                frame_lens=logit_lengths.cpu().numpy(),
                ipa_transcripts=[x['ipa'] for x in batch],
                transcripts=[x['word_segmented'] for x in batch]
            )
        )
        audio_filenames.extend(
            [os.path.basename(x["audio"]["path"]) for x in batch]
        )
    save_textgrids(
        audio_filenames,
        alignments,
        os.path.join(output_path, lang)
    )  

100%|█████████████████████████████████████████████| 1/1 [01:05<00:00, 65.74s/it]
